<a href="https://colab.research.google.com/github/mayuresh-tungare/c3m2/blob/main/mini-2/Mini_Assignment_2_Mayuresh_Tungare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The data set ‘Domestic Flight Delay Records’ contains comprehensive data about domestic flights in the United States, such as flight dates, airtime, flight distance, scheduled departure/arrival times and departure and arrival delays. With the given data set, solve the following tasks using PySpark.

Complete the following tasks using the provided data set:
Read data from here: https://raw.githubusercontent.com/mayuresh-tungare/c3m2/refs/heads/main/mini-2/Flight%20Dataset%20-%20CSV(in).csv

In [6]:
from pyspark.sql import SparkSession
import requests
import tempfile
import os

# Initialize SparkSession
spark = SparkSession.builder.appName("FlightDataAnalysis").getOrCreate()

# URL of the dataset
data_url = "https://raw.githubusercontent.com/mayuresh-tungare/c3m2/refs/heads/main/mini-2/Flight%20Dataset%20-%20CSV(in).csv"

# Download the CSV file to a temporary location
response = requests.get(data_url)
response.raise_for_status() # Raise an exception for bad status codes

# Create a temporary file to store the CSV data
with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.csv', encoding='utf-8') as temp_csv_file:
    temp_csv_file.write(response.text)
    temp_file_path = temp_csv_file.name

# Read the CSV data from the temporary local file into a Spark DataFrame
df = spark.read.csv(temp_file_path, header=True, inferSchema=True)

# Show the schema and a few rows to verify
df.printSchema()
df.show(5)

# The temporary file will not be removed, so it remains available for subsequent Spark actions.

root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)

+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows


Task 1: Create a function that gives back how many flights arrived earlier than expected.

In [7]:
def count_early_arrivals(dataframe):
    """
    Counts the number of flights that arrived earlier than expected.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame with flight data.

    Returns:
        int: The number of flights with a negative arrival delay.
    """
    # Filter for flights where ARR_DELAY is negative (arrived earlier than expected)
    early_arrivals_df = dataframe.filter(dataframe['ARR_DELAY'] < 0)

    # Count the number of such flights
    num_early_arrivals = early_arrivals_df.count()

    return num_early_arrivals

# Example usage:
num_early = count_early_arrivals(df)
print(f"Number of flights that arrived earlier than expected: {num_early}")

Number of flights that arrived earlier than expected: 534655


Task 2: Create a function that determines the typical departure time for flights over 2000 miles.

In [8]:
from pyspark.sql.functions import avg

def typical_departure_time_long_flights(dataframe, min_distance=2000):
    """
    Determines the typical (average) departure time for flights over a specified distance.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame with flight data.
        min_distance (int): The minimum distance in miles to consider for long flights.

    Returns:
        float: The average departure time for flights over the specified distance.
    """
    # Filter for flights over the specified distance
    long_flights_df = dataframe.filter(dataframe['DISTANCE'] > min_distance)

    # Calculate the average departure time
    avg_dep_time = long_flights_df.select(avg('DEP_TIME')).collect()[0][0]

    return avg_dep_time

# Example usage:
avg_dep_time_long = typical_departure_time_long_flights(df, min_distance=2000)
print(f"Typical departure time for flights over 2000 miles: {avg_dep_time_long:.2f}")

Typical departure time for flights over 2000 miles: 13.97


Task 3: Create a function that gives back the proportion of flights that have arrival delays longer than 60 minutes.

In [9]:
def proportion_long_arrival_delays(dataframe, delay_threshold=60):
    """
    Calculates the proportion of flights with arrival delays longer than a specified threshold.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame with flight data.
        delay_threshold (int): The arrival delay threshold in minutes.

    Returns:
        float: The proportion of flights with arrival delays longer than the threshold.
               Returns 0.0 if there are no flights.
    """
    total_flights = dataframe.count()

    if total_flights == 0:
        return 0.0

    # Filter for flights where ARR_DELAY is greater than the threshold
    long_delayed_flights_df = dataframe.filter(dataframe['ARR_DELAY'] > delay_threshold)

    # Count the number of such flights
    num_long_delayed_flights = long_delayed_flights_df.count()

    # Calculate the proportion
    proportion = num_long_delayed_flights / total_flights

    return proportion

# Example usage:
proportion = proportion_long_arrival_delays(df, delay_threshold=60)
print(f"Proportion of flights with arrival delays longer than 60 minutes: {proportion:.4f}")

Proportion of flights with arrival delays longer than 60 minutes: 0.0531


Task 4: Create a function that gives the average airtime for flights that left earlier than 9:00 am.

In [10]:
from pyspark.sql.functions import avg

def average_airtime_early_flights(dataframe, dep_time_threshold=9.0):
    """
    Calculates the average airtime for flights that departed earlier than a specified time.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame with flight data.
        dep_time_threshold (float): The departure time threshold (e.g., 9.0 for 9:00 AM).

    Returns:
        float: The average airtime for flights departing before the threshold.
               Returns None if no such flights are found.
    """
    # Filter for flights that left earlier than the specified time
    early_flights_df = dataframe.filter(dataframe['DEP_TIME'] < dep_time_threshold)

    # Calculate the average airtime for these flights
    # collect()[0][0] is used to extract the scalar value from the result
    avg_airtime = early_flights_df.select(avg('AIR_TIME')).collect()[0][0]

    return avg_airtime

# Example usage:
avg_airtime = average_airtime_early_flights(df, dep_time_threshold=9.0)
print(f"Average airtime for flights departing before 9:00 AM: {avg_airtime:.2f} minutes")

Average airtime for flights departing before 9:00 AM: 111.36 minutes


Task 5: Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure.

In [11]:
from pyspark.sql.functions import max

def max_arrival_delay_on_time_departure(dataframe):
    """
    Determines the maximum arrival delay for flights that did not experience a delay upon departure.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame with flight data.

    Returns:
        int or None: The maximum arrival delay for flights with DEP_DELAY <= 0,
                     or None if no such flights exist or ARR_DELAY is null for all.
    """
    # Filter for flights where DEP_DELAY is 0 or negative (no delay upon departure)
    on_time_departures_df = dataframe.filter(dataframe['DEP_DELAY'] <= 0)

    # Calculate the maximum arrival delay for these flights
    # collect()[0][0] is used to extract the scalar value from the result
    max_delay = on_time_departures_df.select(max('ARR_DELAY')).collect()[0][0]

    return max_delay

# Example usage:
max_arr_delay = max_arrival_delay_on_time_departure(df)
print(f"Maximum arrival delay for flights with no departure delay: {max_arr_delay} minutes")

Maximum arrival delay for flights with no departure delay: 701 minutes
